# M6 evaluation and human-review demo

This notebook evaluates the saved M1-M5 offline pipeline on authorised Drive media. It first reviews/finalizes the Drive manifest, then runs each selected video once or reuses a compatible saved evaluation, replays the five incident strategies from stored signals, matches unique `active` transitions to reviewed events, and writes an auditable report. M3B and the VLM are optional; neither changes incident metrics.

## How to execute this Colab

1. Run `colab-notebooks/dataset_download.ipynb` first if you need evaluation media. Keep media under `CROWD_SAFETY_DATA_ROOT`.
2. Run this notebook top-to-bottom once in a fresh Colab runtime. The first code cell clones/installs the repository and mounts Drive.
3. Use the optional draft-manifest cell only when you need to discover new videos. Run the manifest-review section for every real evaluation; it corrects durations and collects relative violence events before saving the reviewed Drive manifest.
4. Leave `RUN_EVALUATION = False` for media-independent self-checks. For a selected-video smoke run, set it to `True`, set `SELECTED_VIDEO_IDS`, and run the final cell. Set `REUSE_EXISTING_RUN = True` with `EXISTING_EVALUATION_DIR` to skip detector/violence inference and recompute replay/metrics from saved artifacts.
5. Inspect `metrics.json`, `summary.md`, and the saved replay/source evidence before running the full reviewed manifest.

The final execution branch refuses an unfinished violent manifest. Human annotation timestamps are always relative to the beginning of each resolved MP4; no filename timestamp is inferred.


In [2]:
from __future__ import annotations

from copy import deepcopy
from dataclasses import replace
from datetime import datetime, timezone
import hashlib, json, os, platform, shutil, subprocess, sys, tempfile
from pathlib import Path, PurePosixPath
from uuid import uuid4

REPO = Path('/content/realtime-crowd-safety-monitoring')
if not (REPO / 'src').is_dir():
    local_repo = Path.cwd()
    if (local_repo / 'src').is_dir():
        REPO = local_repo
    else:
        if REPO.exists():
            shutil.rmtree(REPO)
        clone = subprocess.run([
            'git', 'clone', '--depth', '1',
            'https://github.com/govardhan-06/realtime-crowd-safety-monitoring.git',
            str(REPO),
        ], capture_output=True, text=True)
        if clone.returncode != 0:
            raise RuntimeError(f'Could not clone repository: {clone.stderr.strip()}')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{REPO}[violence]'], check=True)

DRIVE_ROOT = Path('/content/drive/MyDrive/crowd_safety')
os.environ.setdefault('CROWD_SAFETY_DATA_ROOT', str(DRIVE_ROOT))
os.environ.setdefault('CROWD_SAFETY_EVAL_ROOT', os.path.join(os.environ['CROWD_SAFETY_DATA_ROOT'], 'evaluation', 'runs'))
sys.path.insert(0, str(REPO / 'src'))

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except ImportError:
    pass

DATA_ROOT = Path(os.environ['CROWD_SAFETY_DATA_ROOT']).expanduser()

def _video_duration(path):
    command = [
        'ffprobe', '-v', 'error', '-show_entries', 'format=duration',
        '-of', 'default=noprint_wrappers=1:nokey=1', str(path)
    ]
    try:
        result = subprocess.run(command, capture_output=True, text=True)
    except OSError as exc:
        result = None
        probe_error = str(exc)
    else:
        probe_error = result.stderr.strip()
    if result is not None and result.returncode == 0 and result.stdout.strip():
        return round(float(result.stdout.strip()), 3)
    # Some Colab/Drive video files have a probe-unfriendly container but are
    # still readable by OpenCV. Use it only as a duration fallback.
    try:
        import cv2
        capture = cv2.VideoCapture(str(path))
        fps = capture.get(cv2.CAP_PROP_FPS)
        frames = capture.get(cv2.CAP_PROP_FRAME_COUNT)
        capture.release()
        if fps > 0 and frames > 0:
            return round(frames / fps, 3)
    except Exception:
        pass
    detail = probe_error.splitlines()[-1] if probe_error else 'unknown probe error'
    raise ValueError(f'unreadable video {path}: {detail}')
MANIFEST_ROOT = DATA_ROOT / 'evaluation/manifests'
EVAL_ROOT = Path(os.environ['CROWD_SAFETY_EVAL_ROOT']).expanduser()
MANIFEST_PATH = MANIFEST_ROOT / 'manifest.json'
CONFIG_PATH = REPO / 'configs/pipeline/dev.toml'
STRATEGIES = ('violence-only', 'crowd-only', 'naive-or', 'rule-fusion', 'temporal')
RUN_EVALUATION = False  # change to True only after the reviewed manifest and Drive media are ready
SELECTED_VIDEO_IDS = None  # e.g. ['ubi-001']; keep None for every selected manifest entry
REUSE_EXISTING_RUN = False  # metric/replay-only mode; requires a compatible evaluation directory
EXISTING_EVALUATION_DIR = None  # e.g. DATA_ROOT / 'evaluation/runs/<evaluation-id>'
EXISTING_RUN_MAP = None  # optional {video_id: source-run-directory}; otherwise reuse requires runs.json
print({'repo': str(REPO), 'drive_root': str(DRIVE_ROOT), 'manifest': str(MANIFEST_PATH), 'evaluation_root': str(EVAL_ROOT), 'run_evaluation': RUN_EVALUATION, 'reuse_existing_run': REUSE_EXISTING_RUN})


Mounted at /content/drive
{'repo': '/content/realtime-crowd-safety-monitoring', 'drive_root': '/content/drive/MyDrive/crowd_safety', 'manifest': '/content/realtime-crowd-safety-monitoring/evaluation/manifests/example-test.json', 'evaluation_root': '/content/drive/MyDrive/crowd_safety/evaluation/runs', 'run_evaluation': False, 'reuse_existing_run': False}


## 0. Generate a draft manifest from Drive videos

Run this after `dataset_download.ipynb`. It automates file discovery and duration/model-label metadata, but leaves event timestamps for human review.

In [ ]:
# Keep the manifest local; only the referenced media lives in Drive.
DRAFT_MANIFEST_PATH = REPO / 'evaluation' / 'manifests' / 'manifest-draft.json'
DRAFT_SOURCE_DIRS = ((DATA_ROOT / 'datasets' / 'ubi_fights', 'ubi_fights', 'ubi', None),)
_DRAFT_LABELS = {
    'normal': {'normal', 'nonfight', 'nonviolence', 'nonviolent', 'safe'},
    'violent': {'fight', 'violence', 'violent'},
}

def _draft_model_label(path, dataset):
    matches = {label for label, aliases in _DRAFT_LABELS.items() if any(part.name.casefold().replace('_', '').replace('-', '') in aliases for part in (path, *path.parents))}
    if len(matches) != 1:
        return None
    return matches.pop()

def generate_draft_manifest():
    entries = []
    unreadable = []
    for folder, dataset, prefix, limit in DRAFT_SOURCE_DIRS:
        videos = sorted(folder.rglob('*.mp4')) if folder.is_dir() else []
        if not videos:
            print(f'No MP4 files found: {folder}')
            continue
        if limit is not None:
            videos = videos[:limit]
        for index, path in enumerate(videos, start=1):
            label = _draft_model_label(path, dataset)
            if label not in {'normal', 'violent'}:
                unreadable.append(f'{path}: no approved binary class directory')
                continue
            try:
                duration = _video_duration(path)
            except ValueError as exc:
                unreadable.append(str(exc))
                continue
            entries.append({
                'video_id': f'{prefix}-{index:03d}',
                'media': {'relative_path': str(path.relative_to(DATA_ROOT))},
                'split': 'test',
                'dataset': dataset,
                'source_id': f'{prefix}-camera-{index:03d}',
                'session_id': f'{prefix}-session-{index:03d}',
                'scenario': 'normal' if label != 'violent' else 'staged-violence',
                'duration_s': duration,
                'expected_alert': False,
                'model_label': label,
                'events': [],
                'tags': ['hard-negative'] if label == 'normal' else ['staged'],
            })
    if not entries:
        raise FileNotFoundError('No evaluation videos found; run dataset_download.ipynb first')
    manifest = {
        'schema_version': '1.0',
        'evaluation_id': 'drive-draft',
        'matching': {
            'temporal_tolerance_s': 1.0,
            'require_temporal_overlap': True,
            'actionable_state': 'active',
        },
        'entries': entries,
    }
    DRAFT_MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)
    DRAFT_MANIFEST_PATH.write_text(json.dumps(manifest, indent=2) + '\n')
    print(f'Wrote {len(entries)} draft entries to {DRAFT_MANIFEST_PATH}')
    if unreadable:
        print('Skipped unreadable videos:')
        print('\n'.join(f' - {item}' for item in unreadable))
    if not Path(MANIFEST_PATH).is_file() or Path(MANIFEST_PATH).name == 'example-test.json':
        globals()['MANIFEST_PATH'] = DRAFT_MANIFEST_PATH
    print(f'Active manifest: {MANIFEST_PATH}')
    print('Review positive clips, set expected_alert=true, and add event onset_s/end_s before using this manifest.')
    return manifest

GENERATE_DRAFT_MANIFEST = False
if GENERATE_DRAFT_MANIFEST:
    draft_manifest = generate_draft_manifest()
else:
    print('Draft generation skipped; using active manifest:', MANIFEST_PATH)


## 0.1 Select the manifest and execution mode

The manifest is the evaluation contract. It lists each video, its Drive-relative media path, dataset/split/scenario, and reviewed event intervals. The notebook uses it to resolve authorised media and compare predicted `active` incidents against human-reviewed events; it is not a model input or a replacement for the video files. For real evaluation, review the generated draft, set `expected_alert`, and add each event's `onset_s`, `end_s`, label, severity, source, and ROI where applicable.

The notebook uses this Drive root: `/content/drive/MyDrive/crowd_safety`. The preferred manifest location is `/content/drive/MyDrive/crowd_safety/evaluation/manifests/manifest.json`. If the reviewed JSON is only on your laptop, run the upload cell below, choose the local file in the picker, and it will copy it into that Drive location and make it the active manifest. Colab cannot use `/Users/...` paths directly. Set `RUN_EVALUATION = True` and optionally limit `SELECTED_VIDEO_IDS`; then continue top-to-bottom.


In [14]:
# Set True, then execute this cell in the current Colab browser session to upload a local manifest.
UPLOAD_REVIEWED_MANIFEST = True
if UPLOAD_REVIEWED_MANIFEST:
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Upload exactly one manifest JSON file')
    uploaded_name, uploaded_bytes = next(iter(uploaded.items()))
    if not uploaded_name.lower().endswith('.json'):
        raise ValueError('The uploaded file must be JSON')
    MANIFEST_PATH = MANIFEST_ROOT / 'manifest.json'
    MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)
    MANIFEST_PATH.write_bytes(uploaded_bytes)
    print('Uploaded active manifest:', MANIFEST_PATH)
else:
    print('Local manifest upload skipped')


KeyboardInterrupt: 

In [ ]:
RUN_EVALUATION = False  # enable only after review_manifest() prints: Manifest ready for evaluation: YES


## 1. Manifest contract and safe media resolution

The checked-in JSON schema is the interchange contract. The small validator below keeps the notebook runnable without adding a JSON-schema dependency and enforces the security/data-integrity rules that affect execution.

In [ ]:
SCENARIOS = {'normal', 'dense-crowd', 'staged-violence', 'combined-risk', 'hard-negative'}
EVENT_LABELS = {'crowd-risk', 'violence', 'combined-risk'}
SEVERITIES = {'low', 'medium', 'high', 'critical'}
TAGS = {'dense', 'lighting', 'occlusion', 'camera-motion', 'hard-negative', 'staged'}

def _path_error(value):
    path = PurePosixPath(value)
    return Path(value).is_absolute() or path.is_absolute() or '..' in path.parts

def validate_manifest(manifest):
    if not isinstance(manifest, dict): raise ValueError('manifest must be an object')
    errors = []
    if manifest.get('schema_version') != '1.0': errors.append('unsupported schema_version')
    matching = manifest.get('matching', {})
    if not isinstance(matching, dict): errors.append('matching must be an object'); matching = {}
    if not isinstance(matching.get('temporal_tolerance_s'), (int, float)) or isinstance(matching.get('temporal_tolerance_s'), bool) or matching.get('temporal_tolerance_s', -1) < 0: errors.append('invalid temporal tolerance')
    if matching.get('require_temporal_overlap') is not True: errors.append('temporal overlap must be required')
    entries = manifest.get('entries')
    if not isinstance(entries, list) or not entries: errors.append('entries must be non-empty')
    video_ids, event_ids, split_groups = set(), set(), {}
    for entry in entries or []:
        if not isinstance(entry, dict): errors.append('each entry must be an object'); continue
        video_id = entry.get('video_id')
        if not isinstance(video_id, str) or not video_id or video_id in video_ids: errors.append(f'duplicate/invalid video_id: {video_id}')
        else: video_ids.add(video_id)
        media = entry.get('media', {})
        relative_path = media.get('relative_path') if isinstance(media, dict) else None
        if not isinstance(relative_path, str) or not relative_path or _path_error(relative_path): errors.append(f'unsafe media path: {relative_path}')
        split, group = entry.get('split'), (entry.get('source_id'), entry.get('session_id'))
        if not isinstance(entry.get('dataset'), str) or not entry.get('dataset'): errors.append(f'invalid dataset: {video_id}')
        if split not in {'train', 'validation', 'test'}: errors.append(f'unsupported split: {split}')
        if not all(isinstance(value, str) and value for value in group): errors.append(f'invalid source/session: {group}')
        if all(isinstance(value, str) and value for value in group):
            previous = split_groups.setdefault(group, split)
            if previous != split: errors.append(f'split leakage for source/session: {group}')
        duration = entry.get('duration_s')
        duration_valid = isinstance(duration, (int, float)) and not isinstance(duration, bool) and duration > 0
        if not duration_valid: errors.append(f'invalid duration: {video_id}')
        if entry.get('scenario') not in SCENARIOS: errors.append(f'unsupported scenario: {entry.get("scenario")}')
        if not isinstance(entry.get('expected_alert'), bool): errors.append(f'expected_alert must be boolean: {video_id}')
        if entry.get('model_label') not in {'normal', 'violent', None}: errors.append(f'unsupported model label: {video_id}')
        tags = entry.get('tags', [])
        tags_valid = isinstance(tags, list) and all(isinstance(tag, str) for tag in tags)
        if not tags_valid or any(tag not in TAGS for tag in tags) or (tags_valid and len(tags) != len(set(tags))): errors.append(f'unsupported/duplicate tags: {video_id}')
        events = entry.get('events', [])
        if not isinstance(events, list): errors.append(f'events must be a list: {video_id}'); events = []
        for event in events:
            if not isinstance(event, dict): errors.append(f'event must be an object: {video_id}'); continue
            event_id = event.get('event_id')
            if not isinstance(event_id, str) or not event_id or event_id in event_ids: errors.append(f'duplicate/invalid event_id: {event_id}')
            else: event_ids.add(event_id)
            onset, end = event.get('onset_s'), event.get('end_s')
            if event.get('label') not in EVENT_LABELS: errors.append(f'unsupported event label: {event.get("label")}')
            if not isinstance(onset, (int, float)) or isinstance(onset, bool) or not isinstance(end, (int, float)) or isinstance(end, bool) or not duration_valid or not (0 <= onset < end <= duration): errors.append(f'invalid event interval: {event_id}')
            if not isinstance(event.get('expected_alert'), bool) or event.get('severity') not in SEVERITIES: errors.append(f'invalid event outcome: {event_id}')
        if entry.get('expected_alert') != any(isinstance(event, dict) and event.get('expected_alert') for event in events): errors.append(f'entry/event expected_alert mismatch: {video_id}')
    if errors: raise ValueError('; '.join(errors))
    return manifest

def resolve_media(entry, data_root, require_exists=True):
    relative = entry['media']['relative_path']
    if _path_error(relative): raise ValueError(f'media path must be relative and traversal-free: {relative}')
    root, candidate = Path(data_root).expanduser().resolve(), (Path(data_root) / relative).expanduser().resolve()
    try: candidate.relative_to(root)
    except ValueError as exc: raise ValueError(f'media path escapes data root: {relative}') from exc
    if require_exists and not candidate.is_file(): raise FileNotFoundError(f'manifest media is missing: {candidate}')
    return candidate

def load_manifest(path=MANIFEST_PATH, data_root=DATA_ROOT, require_files=False):
    manifest = json.loads(Path(path).read_text())
    validate_manifest(manifest)
    for entry in manifest['entries']: resolve_media(entry, data_root, require_exists=require_files)
    return manifest

## Update deterministic manifest metadata

Run this section after Drive is mounted and before evaluation. It resolves every manifest entry through the safe media path, corrects only `duration_s` from local media (`ffprobe` first, OpenCV fallback), and saves the structurally validated manifest to:

`/content/drive/MyDrive/crowd_safety/evaluation/manifests/manifest.json`

Watch the videos separately and manually edit `expected_alert` and `events` in this JSON. This block does not render videos, prompt for annotations, or change labels/events. Evaluation still refuses to run until the manual annotations are complete.


In [15]:
import hashlib
REVIEWED_MANIFEST_PATH = DATA_ROOT / 'evaluation/manifests/manifest.json'
RUN_MANIFEST_REVIEW = True

print("Reviewing manifest:", REVIEWED_MANIFEST_PATH)

if '_video_duration' not in globals():
    def _video_duration(path):
        command = ['ffprobe', '-v', 'error', '-show_entries', 'format=duration', '-of', 'default=noprint_wrappers=1:nokey=1', str(path)]
        try:
            result = subprocess.run(command, capture_output=True, text=True)
        except OSError as exc:
            result = None
            probe_error = str(exc)
        else:
            probe_error = result.stderr.strip()
        if result is not None and result.returncode == 0 and result.stdout.strip():
            return round(float(result.stdout.strip()), 3)
        try:
            import cv2
            capture = cv2.VideoCapture(str(path))
            fps, frames = capture.get(cv2.CAP_PROP_FPS), capture.get(cv2.CAP_PROP_FRAME_COUNT)
            capture.release()
            if fps > 0 and frames > 0:
                return round(frames / fps, 3)
        except Exception:
            pass
        detail = probe_error.splitlines()[-1] if probe_error else 'unknown probe error'
        raise ValueError(f'unreadable video {path}: {detail}')

if 'resolve_media' not in globals():
    def resolve_media(entry, data_root, require_exists=True):
        relative = entry['media']['relative_path']
        relative_path = PurePosixPath(relative)
        if Path(relative).is_absolute() or relative_path.is_absolute() or '..' in relative_path.parts:
            raise ValueError(f'media path must be relative and traversal-free: {relative}')
        root = Path(data_root).expanduser().resolve()
        candidate = (Path(data_root) / relative).expanduser().resolve()
        try:
            candidate.relative_to(root)
        except ValueError as exc:
            raise ValueError(f'media path escapes data root: {relative}') from exc
        if require_exists and not candidate.is_file():
            raise FileNotFoundError(f'manifest media is missing: {candidate}')
        return candidate

if 'SEVERITIES' not in globals():
    SEVERITIES = {'low', 'medium', 'high', 'critical'}

def _review_sidecar(path):
    path = Path(path)
    return path.with_name(path.name + '.review.json')

def has_review_provenance(path, manifest=None):
    path = Path(path)
    sidecar = _review_sidecar(path)
    if not path.is_file() or not sidecar.is_file():
        return False
    try:
        record = json.loads(sidecar.read_text())
    except (OSError, ValueError):
        return False
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    return record.get('manifest_sha256') == digest and record.get('manifest_path') == str(path)


def manifest_readiness_errors(manifest):
    errors = []
    for entry in manifest.get('entries', []):
        label = entry.get('model_label')
        if label not in {'normal', 'violent'}:
            errors.append(f"{entry.get('video_id')}: reviewed model_label must be normal or violent")
        elif label == 'violent':
            events = [event for event in entry.get('events', []) if isinstance(event, dict) and event.get('expected_alert')]
            if entry.get('expected_alert') is not True or not events:
                errors.append(f"{entry.get('video_id')}: violent clips require at least one expected-alert event")
        elif label == 'normal' and (entry.get('expected_alert') is not False or entry.get('events') != []):
            errors.append(f"{entry.get('video_id')}: normal clips must have expected_alert=false and no events")
    return errors

def require_manifest_ready(manifest):
    errors = manifest_readiness_errors(manifest)
    if errors:
        raise ValueError('Manifest review required before evaluation:\n - ' + '\n - '.join(errors))
    return manifest

def review_manifest(manifest_path=MANIFEST_PATH, data_root=DATA_ROOT, output_path=REVIEWED_MANIFEST_PATH,
                    duration_fn=_video_duration):
    raw = json.loads(Path(manifest_path).read_text())
    if not isinstance(raw, dict) or not isinstance(raw.get('entries'), list):
        raise ValueError('manifest review requires an object with an entries list')
    reviewed = deepcopy(raw)
    corrected = 0
    total_duration = 0.0
    for entry in reviewed['entries']:
        media = resolve_media(entry, data_root, require_exists=True)
        old_duration = entry.get('duration_s')
        duration = float(duration_fn(media))
        if duration <= 0:
            raise ValueError(f'{entry.get("video_id")}: media duration must be positive')
        entry['duration_s'] = round(duration, 3)
        total_duration += duration
        if old_duration is None or abs(float(old_duration) - duration) > 0.01:
            corrected += 1
            print(f'{entry["video_id"]}: duration {old_duration} -> {entry["duration_s"]} s')
    if 'validate_manifest' in globals():
        validate_manifest(reviewed)
    else:
        print('Manifest contract validator not loaded; run the Manifest contract cell above for structural validation.')
    ready = not manifest_readiness_errors(reviewed) if 'manifest_readiness_errors' in globals() else False
    print(f'\nManifest metadata update: total videos={len(reviewed["entries"])} corrected durations={corrected} total camera duration_s={round(total_duration, 3)}')
    print('Manifest ready for evaluation:', 'YES' if ready else 'NO')
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(json.dumps(reviewed, indent=2, sort_keys=True) + '\n')
    _review_sidecar(output_path).write_text(json.dumps({
        'manifest_path': str(output_path),
        'manifest_sha256': hashlib.sha256(output_path.read_bytes()).hexdigest(),
        'reviewed_at': datetime.now(timezone.utc).isoformat(),
    }, indent=2, sort_keys=True) + '\n')
    globals()['MANIFEST_PATH'] = output_path
    globals()['MANIFEST_REVIEWED'] = True
    print('Saved reviewed manifest:', output_path)
    return reviewed

RUN_MANIFEST_REVIEW = True # update deterministic durations; annotate events manually in the manifest
if RUN_MANIFEST_REVIEW:
    reviewed_manifest = review_manifest()
else:
    print('Manifest review paused; set RUN_MANIFEST_REVIEW = True before evaluation')


Reviewing manifest: /content/drive/MyDrive/crowd_safety/evaluation/manifests/manifest.json


FileNotFoundError: manifest media is missing: /content/drive/MyDrive/crowd_safety/fixtures/positive-violent.mp4

In [ ]:
# Manifest and review self-checks are media-independent.
fixture = load_manifest(REPO / 'evaluation/manifests/example-test.json', require_files=False)
assert len(fixture['entries']) == 3 and len(fixture['entries'][2]['events']) == 2

def must_reject(mutated):
    try:
        validate_manifest(mutated)
    except ValueError:
        return
    raise AssertionError('invalid manifest was accepted')

bad = deepcopy(fixture); bad['entries'][1]['video_id'] = bad['entries'][0]['video_id']; must_reject(bad)
bad = deepcopy(fixture); bad['entries'][0]['events'][0]['end_s'] = 2.0; must_reject(bad)
bad = deepcopy(fixture); bad['entries'][0]['media']['relative_path'] = '../outside.mp4'; must_reject(bad)
bad = deepcopy(fixture); bad['entries'][0]['events'][0]['label'] = 'unsupported'; must_reject(bad)
bad = deepcopy(fixture); bad['entries'][0]['events'] = [None]; must_reject(bad)
bad = deepcopy(fixture); bad['entries'][1]['source_id'] = bad['entries'][0]['source_id']; bad['entries'][1]['session_id'] = bad['entries'][0]['session_id']; bad['entries'][1]['split'] = 'train'; must_reject(bad)
try:
    require_manifest_ready({'entries': [{'video_id': 'unreviewed', 'model_label': None, 'expected_alert': False, 'events': []}]})
except ValueError: pass
else: raise AssertionError('unknown model label accepted as reviewed')

with tempfile.TemporaryDirectory() as tmp:
    root = Path(tmp)
    media = root / 'clip.mp4'
    media.write_bytes(b'fixture')
    manifest = {
        'schema_version': '1.0', 'evaluation_id': 'review-fixture',
        'matching': {'temporal_tolerance_s': 1.0, 'require_temporal_overlap': True, 'actionable_state': 'active'},
        'entries': [
            {'video_id': 'violent', 'media': {'relative_path': 'clip.mp4'}, 'split': 'test', 'dataset': 'fixture',
             'source_id': 'camera-v', 'session_id': 'session-v', 'scenario': 'staged-violence', 'duration_s': 1.0,
             'expected_alert': False, 'model_label': 'violent', 'events': [], 'tags': ['staged']},
            {'video_id': 'normal', 'media': {'relative_path': 'clip.mp4'}, 'split': 'test', 'dataset': 'fixture',
             'source_id': 'camera-n', 'session_id': 'session-n', 'scenario': 'normal', 'duration_s': 1.0,
             'expected_alert': True, 'model_label': 'normal',
             'events': [{'event_id': 'stale', 'label': 'violence', 'onset_s': 0.1, 'end_s': 0.2, 'expected_alert': True, 'severity': 'low'}], 'tags': []},
        ],
    }
    source = root / 'draft.json'
    source.write_text(json.dumps(manifest))
    reviewed = review_manifest(source, root, root / 'reviewed.json', duration_fn=lambda _: 12.0)
    assert reviewed['entries'][0]['duration_s'] == 12.0
    assert reviewed['entries'][0]['events'] == [] and reviewed['entries'][0]['expected_alert'] is False
    assert reviewed['entries'][1]['expected_alert'] is True and len(reviewed['entries'][1]['events']) == 1
    assert json.loads((root / 'reviewed.json').read_text()) == reviewed

    incomplete = deepcopy(manifest)
    incomplete['entries'] = [manifest['entries'][0]]
    incomplete_path = root / 'incomplete.json'
    incomplete_path.write_text(json.dumps(incomplete))
    incomplete_reviewed = review_manifest(incomplete_path, root, root / 'reviewed-incomplete.json', duration_fn=lambda _: 12.0)
    assert incomplete_reviewed['entries'][0]['duration_s'] == 12.0
    assert manifest_readiness_errors(incomplete_reviewed)

print('manifest and review self-checks passed')


## 2. Prediction extraction, one-to-one event matching, and metrics

In [ ]:
def jsonl(path):
    path = Path(path)
    return [json.loads(line) for line in path.read_text().splitlines() if line.strip()] if path.exists() else []

def _source_evidence_manifests(evidence_root, run_id):
    root = Path(evidence_root) / run_id
    return [json.loads(path.read_text()) for path in sorted(root.glob('*/manifest.json'))] if root.is_dir() else []

def _intervals_overlap(start_a, end_a, start_b, end_b, tolerance=0.0):
    start_a, end_a = float(start_a), float(end_a)
    start_b, end_b = float(start_b), float(end_b)
    if start_a == end_a:
        return start_b - tolerance <= start_a <= end_b + tolerance
    if start_b == end_b:
        return start_a - tolerance <= start_b <= end_a + tolerance
    return max(start_a, start_b - tolerance) < min(end_a, end_b + tolerance)

def _evidence_status(evidence_root, run_id, prediction):
    source_ids = {prediction.get('source_id'), prediction.get('observed_source_id')} - {None, ''}
    candidates = [manifest for manifest in _source_evidence_manifests(evidence_root, run_id)
                  if manifest.get('source_id') in source_ids and _intervals_overlap(
                      prediction['predicted_onset_s'], prediction['predicted_end_s'],
                      manifest.get('start_s', -1), manifest.get('end_s', -1))]
    if len(candidates) != 1:
        return 'not_comparable'
    return 'available' if any(item.get('status') == 'available' for item in candidates[0].get('artifacts', [])) else 'not_comparable'

def active_alerts(video_id, strategy, replay_dir, source_run_dir, evidence_root, logical_source_id=None, evidence_run_id=None):
    snapshots = {row['incident_id']: row for row in jsonl(Path(replay_dir) / strategy / 'incidents.jsonl')}
    alerts, seen_incidents = [], set()
    for transition in jsonl(Path(replay_dir) / strategy / 'transitions.jsonl'):
        incident_id = transition.get('incident_id')
        if transition.get('to_state') != 'active' or incident_id in seen_incidents:
            continue
        seen_incidents.add(incident_id)
        incident = snapshots.get(incident_id, {})
        prediction = {
            'prediction_id': f"{incident_id}:{float(transition['timestamp_s']):.6f}",
            'video_id': video_id, 'strategy': strategy, 'incident_id': incident_id,
            'source_id': logical_source_id or transition['source_id'], 'observed_source_id': transition['source_id'], 'roi': transition['region_id'],
            'predicted_onset_s': float(transition['timestamp_s']),
            'predicted_end_s': max(float(transition['timestamp_s']), float(incident.get('last_updated_at_s', transition['timestamp_s']))),
            'severity': transition.get('severity'), 'peak_risk': incident.get('peak_risk'),
            'reason_codes': incident.get('reason_codes', transition.get('reason_codes', [])),
        }
        prediction['evidence_status'] = _evidence_status(evidence_root, evidence_run_id or source_run_dir.name, prediction)
        prediction['evidence_complete'] = prediction['evidence_status'] == 'available'
        alerts.append(prediction)
    return alerts

def _compatible(prediction, event):
    return ((not event.get('source_id') or prediction['source_id'] == event['source_id']) and
            (not event.get('roi') or prediction['roi'] == event['roi']))

def _overlaps(prediction, event, tolerance):
    return _intervals_overlap(prediction['predicted_onset_s'], prediction['predicted_end_s'], event['onset_s'], event['end_s'], tolerance)

def match_events(entry, predictions, matching):
    tolerance = float(matching['temporal_tolerance_s'])
    expected = sorted((event for event in entry['events'] if event['expected_alert']), key=lambda item: item['onset_s'])
    remaining = list(sorted(predictions, key=lambda item: item['predicted_onset_s']))
    matched, matched_event_ids = [], set()
    for event in expected:
        candidates = [item for item in remaining if _compatible(item, event) and _overlaps(item, event, tolerance)]
        if not candidates: continue
        prediction = candidates[0]; remaining.remove(prediction); matched_event_ids.add(event['event_id'])
        matched.append({'event_id': event['event_id'], 'prediction_id': prediction['prediction_id'], 'delay_s': prediction['predicted_onset_s'] - event['onset_s'], 'duration_error_s': prediction['predicted_end_s'] - event['end_s']})
    duplicate_ids = {item['prediction_id'] for item in remaining if any(event['event_id'] in matched_event_ids and _compatible(item, event) and _overlaps(item, event, tolerance) for event in expected)}
    return {'matched': matched, 'duplicates': len(duplicate_ids), 'false_alerts': len(remaining) - len(duplicate_ids), 'missed': len(expected) - len(matched)}

def aggregate_strategy(entries, predictions_by_video, strategy, matching):
    camera_hours = sum(float(entry['duration_s']) for entry in entries) / 3600.0
    result = {'strategy': strategy, 'expected_events': 0, 'alerts': 0, 'true_positives': 0, 'false_alerts': 0, 'duplicates': 0, 'missed_events': 0, 'detection_delays_s': [], 'duration_errors_s': [], 'evidence_complete_alerts': 0, 'entry_results': []}
    evidence_statuses = []
    for entry in entries:
        predictions = predictions_by_video.get(entry['video_id'], [])
        outcome = match_events(entry, predictions, matching)
        result['expected_events'] += sum(event['expected_alert'] for event in entry['events'])
        result['alerts'] += len(predictions); result['true_positives'] += len(outcome['matched'])
        result['false_alerts'] += outcome['false_alerts']; result['duplicates'] += outcome['duplicates']; result['missed_events'] += outcome['missed']
        result['detection_delays_s'] += [item['delay_s'] for item in outcome['matched']]
        result['duration_errors_s'] += [item['duration_error_s'] for item in outcome['matched']]
        result['evidence_complete_alerts'] += sum(item['evidence_complete'] for item in predictions)
        evidence_statuses += [item['evidence_status'] for item in predictions]
        result['entry_results'].append({'video_id': entry['video_id'], 'dataset': entry['dataset'], 'source_id': entry['source_id'], 'scenario': entry['scenario'], 'tags': entry['tags'], **outcome, 'alert_count': len(predictions)})
    tp, fp, fn = result['true_positives'], result['false_alerts'], result['missed_events']
    result.update({'precision': tp / (tp + fp) if tp + fp else 0.0, 'recall': tp / (tp + fn) if tp + fn else 0.0,
        'f1': 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else 0.0,
        'camera_hours': camera_hours, 'false_alerts_per_camera_hour': fp / camera_hours if camera_hours else 0.0,
        'duplicate_alerts_per_true_event': result['duplicates'] / result['expected_events'] if result['expected_events'] else 0.0,
        'alert_rate_per_camera_hour': result['alerts'] / camera_hours if camera_hours else 0.0,
        'detection_delay_s_mean': sum(result['detection_delays_s']) / len(result['detection_delays_s']) if result['detection_delays_s'] else None,
        'duration_error_s_mean': sum(result['duration_errors_s']) / len(result['duration_errors_s']) if result['duration_errors_s'] else None,
        'evidence_completeness': result['evidence_complete_alerts'] / result['alerts'] if evidence_statuses and all(status == 'available' for status in evidence_statuses) else None,
        'evidence_completeness_status': 'not_comparable' if any(status != 'available' for status in evidence_statuses) else ('available' if evidence_statuses else 'unavailable')})
    return result


In [ ]:
# Alert matching self-check: match, miss, false alert, duplicate, ROI mismatch, and delay.
test_entry = {'video_id': 'synthetic', 'source_id': 'camera', 'session_id': 's', 'scenario': 'combined-risk', 'duration_s': 10, 'expected_alert': True, 'events': [{'event_id': 'e1', 'label': 'combined-risk', 'onset_s': 3, 'end_s': 5, 'expected_alert': True, 'severity': 'high', 'source_id': 'camera', 'roi': 'main'}, {'event_id': 'e2', 'label': 'violence', 'onset_s': 7, 'end_s': 8, 'expected_alert': True, 'severity': 'medium', 'source_id': 'camera', 'roi': 'side'}], 'tags': [], '_matching': {'temporal_tolerance_s': 1, 'require_temporal_overlap': True}}
prediction = {'prediction_id': 'p1', 'source_id': 'camera', 'roi': 'main', 'predicted_onset_s': 4, 'predicted_end_s': 6, 'evidence_complete': True}
duplicate = {**prediction, 'prediction_id': 'p2', 'predicted_onset_s': 4.5}
false_alert = {**prediction, 'prediction_id': 'p3', 'roi': 'other', 'predicted_onset_s': 9, 'predicted_end_s': 9.5}
outcome = match_events(test_entry, [prediction, duplicate, false_alert], test_entry['_matching'])
assert len(outcome['matched']) == 1 and outcome['duplicates'] == 1 and outcome['false_alerts'] == 1 and outcome['missed'] == 1
assert outcome['matched'][0]['delay_s'] == 1
assert not _intervals_overlap(0, 2, 2, 4, 0.0)
assert _intervals_overlap(0, 2, 2, 4, 0.1)

with tempfile.TemporaryDirectory() as tmp:
    root = Path(tmp); replay = root / 'replay' / 'temporal'; replay.mkdir(parents=True); evidence = root / 'evidence' / 'source-run' / 'source-incident'; evidence.mkdir(parents=True)
    incidents = [{'incident_id': 'replay-incident', 'last_updated_at_s': 6.0, 'peak_risk': 0.8, 'reason_codes': ['violence']}]
    transitions = [
        {'incident_id': 'replay-incident', 'source_id': 'camera', 'region_id': 'zone-main', 'timestamp_s': 2.0, 'to_state': 'active', 'severity': 'high'},
        {'incident_id': 'replay-incident', 'source_id': 'camera', 'region_id': 'zone-main', 'timestamp_s': 4.0, 'to_state': 'resolving', 'severity': 'medium'},
        {'incident_id': 'replay-incident', 'source_id': 'camera', 'region_id': 'zone-main', 'timestamp_s': 5.0, 'to_state': 'active', 'severity': 'high'},
        {'incident_id': 'new-incident', 'source_id': 'camera', 'region_id': 'zone-main', 'timestamp_s': 8.0, 'to_state': 'active', 'severity': 'high'},
    ]
    (replay / 'incidents.jsonl').write_text(''.join(json.dumps(row)+'\n' for row in incidents + [{'incident_id': 'new-incident', 'last_updated_at_s': 9.0, 'peak_risk': 0.9, 'reason_codes': []}]))
    (replay / 'transitions.jsonl').write_text(''.join(json.dumps(row)+'\n' for row in transitions))
    (evidence / 'manifest.json').write_text(json.dumps({'run_id': 'source-run', 'incident_id': 'source-incident', 'source_id': 'camera', 'start_s': 1.5, 'end_s': 6.5, 'artifacts': [{'status': 'available'}]}))
    alerts = active_alerts('synthetic', 'temporal', root / 'replay', root / 'source-run', root / 'evidence')
    assert len(alerts) == 2 and alerts[0]['evidence_status'] == 'available'
    assert len(jsonl(replay / 'transitions.jsonl')) == 4
print('matcher, unique-alert, and ID-independent evidence self-checks passed')


## 3. Run once, replay five strategies, and write the report

In [ ]:
from crowd_safety.artifacts import config_hash, resolved_config
from crowd_safety.config import load_config
from crowd_safety.persistence import MemoryPersistence, import_run
from crowd_safety.replay import replay_run
from crowd_safety.runner import process_video

M3A_X3D_PROVENANCE = {
    'backend': 'x3d', 'repository': 'visionlab-ai/school-violence-detection-models',
    'checkpoint': 'final/final_x3d_realtime.pt', 'revision': 'a744b6af7496f0cbfa4f0ba32acd46b65e52d4e1',
    'architecture': 'x3d_m', 'sample_count': 16, 'labels': ['non-violent', 'violent'],
    'checkpoint_sha256': 'e833f69d110f167cad4a6c38d385564bdb2f6de63d246e45cb03ff9aa17f0349',
}

def write_jsonl(path, rows):
    Path(path).write_text(''.join(json.dumps(row, sort_keys=True) + '\n' for row in rows))

def stage_health_summary(source_runs):
    return {str(run['video_id']): json.loads((Path(run['source_run']) / 'metrics.json').read_text()).get('stage_health', {}) for run in source_runs if run['status'] == 'success'}

def model_predictions_m3a(source_runs, threshold):
    rows = []
    for run in source_runs:
        if run['status'] != 'success':
            continue
        evidence_rows = jsonl(Path(run['source_run']) / 'violence.jsonl')
        meta = json.loads((Path(run['source_run']) / 'metadata.json').read_text())
        provenance = meta.get('provenance') or {}
        for item in evidence_rows:
            evidence = item.get('evidence', {})
            score = evidence.get('score')
            status = evidence.get('status', 'unavailable')
            rows.append({
                'model': 'M3A', 'video_id': run['video_id'],
                'clip_start_s': evidence.get('clip_start_s'), 'clip_end_s': evidence.get('clip_end_s'),
                'timestamp_s': item.get('timestamp_s', evidence.get('clip_end_s')),
                'score': float(score) if score is not None else None, 'threshold': threshold,
                'predicted_label': (score is not None and float(score) >= threshold) if status == 'available' else None,
                'label': ('violent' if float(score) >= threshold else 'normal') if status == 'available' and score is not None else None,
                'checkpoint': provenance.get('violence_model', evidence.get('model', 'unknown')),
                'revision': provenance.get('violence_revision', evidence.get('revision')), 'status': status,
                'latency_ms': evidence.get('latency_ms'),
            })
    return rows

def _violence_events(entry):
    return [event for event in entry.get('events', []) if event.get('expected_alert') and event.get('label') in {'violence', 'combined-risk'}]

def _window_target(row, entry, tolerance):
    return any(_intervals_overlap(row.get('clip_start_s'), row.get('clip_end_s'), event['onset_s'], event['end_s'], tolerance) for event in _violence_events(entry))

def _binary_metrics(rows, targets, threshold):
    tp = sum(float(row['score']) >= threshold and target for row, target in zip(rows, targets))
    tn = sum(float(row['score']) < threshold and not target for row, target in zip(rows, targets))
    fp = sum(float(row['score']) >= threshold and not target for row, target in zip(rows, targets))
    fn = sum(float(row['score']) < threshold and target for row, target in zip(rows, targets))
    return {'tp': tp, 'tn': tn, 'fp': fp, 'fn': fn,
            'precision': tp / (tp + fp) if tp + fp else 0.0,
            'recall': tp / (tp + fn) if tp + fn else 0.0,
            'f1': 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else 0.0}

def score_model_predictions(rows, entries, matching=None):
    matching = matching or {'temporal_tolerance_s': 0.0}
    tolerance = float(matching.get('temporal_tolerance_s', 0.0))
    by_video = {entry['video_id']: entry for entry in entries}
    window_rows = [row for row in rows if row.get('clip_start_s') is not None and row.get('clip_end_s') is not None and row.get('score') is not None and row.get('status') == 'available' and row.get('video_id') in by_video]
    threshold = float(next((row.get('threshold') for row in rows if row.get('threshold') is not None), 0.5))
    targets = [_window_target(row, by_video[row['video_id']], tolerance) for row in window_rows]
    counts = _binary_metrics(window_rows, targets, threshold) if window_rows else {'tp': 0, 'tn': 0, 'fp': 0, 'fn': 0, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0}
    pr_curve, roc_curve = [], []
    for curve_threshold in [index / 10 for index in range(11)]:
        curve = _binary_metrics(window_rows, targets, curve_threshold) if window_rows else {'tp': 0, 'tn': 0, 'fp': 0, 'fn': 0, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0}
        pr_curve.append({'threshold': curve_threshold, 'precision': curve['precision'], 'recall': curve['recall']})
        roc_curve.append({'threshold': curve_threshold, 'tpr': curve['recall'], 'fpr': curve['fp'] / (curve['fp'] + curve['tn']) if curve['fp'] + curve['tn'] else 0.0})
    event_rows = []
    for entry in entries:
        for event in _violence_events(entry):
            matches = [row for row in window_rows if row['video_id'] == entry['video_id'] and float(row['score']) >= threshold and _intervals_overlap(row['clip_start_s'], row['clip_end_s'], event['onset_s'], event['end_s'], tolerance)]
            event_rows.append({'event_id': event['event_id'], 'video_id': entry['video_id'], 'detected': bool(matches), 'first_detection_delay_s': max(0.0, min(float(row.get('timestamp_s', row['clip_end_s'])) for row in matches) - float(event['onset_s'])) if matches else None})
    positives = len(event_rows)
    negative_windows = counts['tn'] + counts['fp']
    event_status = 'available' if positives and window_rows else ('unavailable-no-compatible-windows' if positives else 'invalid-no-positive-events')
    event_metrics = {'status': event_status, 'events': event_rows, 'recall': sum(item['detected'] for item in event_rows) / positives if positives else None,
                     'first_detection_delay_s_mean': sum(item['first_detection_delay_s'] for item in event_rows if item['first_detection_delay_s'] is not None) / sum(item['first_detection_delay_s'] is not None for item in event_rows) if any(item['first_detection_delay_s'] is not None for item in event_rows) else None}
    diagnostic = []
    for video_id, video_rows in __import__('itertools').groupby(sorted(window_rows, key=lambda row: row['video_id']), key=lambda row: row['video_id']):
        scores = [float(row['score']) for row in video_rows]
        maximum = max(scores)
        diagnostic.append({'video_id': video_id, 'max_score': maximum, 'predicted_label': maximum >= threshold})
    return {'samples': len(window_rows), **counts, 'confusion_matrix': [[counts['tn'], counts['fp']], [counts['fn'], counts['tp']]],
            'pr_curve': pr_curve, 'roc_curve': roc_curve, 'status': 'available' if window_rows else 'pending-compatible-scores',
            'window_metrics': {'status': 'available' if window_rows else 'pending-compatible-scores', **counts, 'false_positive_window_rate': counts['fp'] / negative_windows if negative_windows else None},
            'event_metrics': event_metrics, 'whole_video_diagnostic': {'status': 'secondary', 'rows': diagnostic}}

def vlm_disabled_record(incident_id):
    return {'schema_version': '1.0', 'incident_id': incident_id, 'status': 'disabled', 'provider': 'disabled', 'model': '', 'text': '', 'grounded': None, 'contradicts_reasons': None, 'unsupported_details': [], 'latency_ms': None, 'reviewer_note': 'VLM disabled; deterministic incident evidence remains authoritative.'}

def markdown_table(rows, columns):
    if not rows: return '_none_'
    header = '| ' + ' | '.join(columns) + ' |\n| ' + ' | '.join('---' for _ in columns) + ' |'
    return header + '\n' + '\n'.join('| ' + ' | '.join(str(row.get(column, '')) for column in columns) + ' |' for row in rows)

def failure_slice_rows(strategy_metrics, failures):
    rows = []
    for strategy, metric in strategy_metrics.items():
        for item in metric['entry_results']:
            category = 'missed_event' if item['missed'] else ('duplicate_alert' if item['duplicates'] else ('false_alert' if item['false_alerts'] else 'no_failure'))
            rows.append({'strategy': strategy, 'video_id': item['video_id'], 'dataset': item['dataset'], 'source_id': item['source_id'], 'scenario': item['scenario'], 'tags': ','.join(item['tags']) or '-', 'category': category, 'alerts': item['alert_count'], 'missed': item['missed'], 'false_alerts': item['false_alerts'], 'duplicates': item['duplicates'], 'error': ''})
    rows.extend({'strategy': '-', 'video_id': failure['video_id'], 'dataset': '-', 'source_id': '-', 'scenario': failure.get('scenario', '-'), 'tags': '-', 'category': 'run_failure', 'alerts': 0, 'missed': '-', 'false_alerts': '-', 'duplicates': '-', 'error': failure.get('error', '')} for failure in failures)
    return rows

def render_report(eval_dir, manifest, strategy_metrics, model_rows, model_status, vlm_rows, failures, failure_slices, latency_rows, integrity):
    template = (REPO / 'evaluation/templates/summary.md').read_text()
    strategy_rows = [{key: metric.get(key) for key in ('strategy', 'expected_events', 'alerts', 'precision', 'recall', 'f1', 'false_alerts_per_camera_hour', 'duplicate_alerts_per_true_event', 'detection_delay_s_mean', 'duration_error_s_mean', 'evidence_completeness', 'evidence_completeness_status')} for metric in strategy_metrics.values()]
    model_metrics = [{'model': name, **score_model_predictions(rows, manifest['entries'], manifest['matching'])} for name, rows in model_rows.items()]
    if model_status.startswith('pending'):
        model_metrics.append({'model': 'M3B', 'status': model_status})
    failure_text = markdown_table(failure_slices, ['strategy', 'video_id', 'dataset', 'source_id', 'scenario', 'tags', 'category', 'alerts', 'missed', 'false_alerts', 'duplicates', 'error'])
    strategy_text = markdown_table(strategy_rows, ['strategy', 'expected_events', 'alerts', 'precision', 'recall', 'f1', 'false_alerts_per_camera_hour', 'duplicate_alerts_per_true_event', 'detection_delay_s_mean', 'duration_error_s_mean', 'evidence_completeness', 'evidence_completeness_status'])
    model_text = markdown_table(model_metrics, ['model', 'status', 'samples', 'precision', 'recall', 'f1', 'confusion_matrix'])
    m3a = next((item for item in model_metrics if item.get('model') == 'M3A'), {})
    event_metrics = m3a.get('event_metrics', {})
    window_metrics = m3a.get('window_metrics', {})
    m3a_text = (f"Window metrics: status={window_metrics.get('status')}, TP={window_metrics.get('tp')}, TN={window_metrics.get('tn')}, FP={window_metrics.get('fp')}, FN={window_metrics.get('fn')}, precision={window_metrics.get('precision')}, recall={window_metrics.get('recall')}, F1={window_metrics.get('f1')}.\n\n"
                 f"Violence-event metrics: status={event_metrics.get('status')}, recall={event_metrics.get('recall')}, first-detection-delay-mean-s={event_metrics.get('first_detection_delay_s_mean')}.\n\n"
                 "Whole-video max score is retained only as a secondary diagnostic in metrics.json; it is not the primary M3A result.")
    vlm_text = f"{len(vlm_rows)} review records; disabled/unavailable VLM status is supplementary and excluded from incident metrics."
    latency_text = markdown_table(latency_rows, ['video_id', 'total_seconds', 'effective_fps', 'violence_seconds', 'fusion_seconds'])
    integrity_text = '\n'.join(f'- {key}: {value}' for key, value in integrity.items())
    limitations = 'M3B is pending unless CROWD_SAFETY_M3B_PREDICTIONS points to a compatible held-out export. Results are limited by the selected annotations and remain offline engineering evidence.'
    values = {'evaluation_id': manifest['evaluation_id'], 'generated_at': datetime.now(timezone.utc).isoformat(), 'entry_count': len(manifest['entries']), 'camera_hours': round(sum(float(entry['duration_s']) for entry in manifest['entries']) / 3600, 4), 'vlm_status': 'disabled', 'm6b_status': model_status, 'strategy_table': strategy_text, 'model_table': model_text, 'm3a_summary': m3a_text, 'integrity_summary': integrity_text, 'failure_slices': failure_text, 'latency_summary': latency_text, 'vlm_review': vlm_text, 'limitations': limitations}
    for key, value in values.items():
        template = template.replace('{{' + key + '}}', str(value))
    (Path(eval_dir) / 'summary.md').write_text(template)
    return model_metrics

def _validate_reuse_source_run(run_directory, entry, data_root=DATA_ROOT):
    run_directory = Path(run_directory)
    config_payload = json.loads((run_directory / 'config.json').read_text())
    values = config_payload.get('config', config_payload)
    stored_input = values.get('input_path') if isinstance(values, dict) else None
    expected_input = resolve_media(entry, data_root, require_exists=False)
    if not isinstance(stored_input, str) or Path(stored_input).expanduser().resolve() != expected_input:
        raise ValueError(f"reuse source/media mismatch for {entry['video_id']}")
    metadata = json.loads((run_directory / 'metadata.json').read_text())
    actual = metadata.get('provenance', {}).get('violence_provenance')
    if not isinstance(actual, dict) or any(actual.get(key) != value for key, value in M3A_X3D_PROVENANCE.items()):
        raise ValueError(f"reuse source/model provenance mismatch for {entry['video_id']}")

def _validate_reuse_manifest(eval_dir, selected, data_root=DATA_ROOT):
    stored_path = Path(eval_dir) / 'manifest.json'
    if not stored_path.is_file():
        raise ValueError(f'reuse evaluation is missing its manifest: {stored_path}')
    stored = json.loads(stored_path.read_text())
    stored_by_id = {entry.get('video_id'): entry for entry in stored.get('entries', [])}
    for entry in selected:
        previous = stored_by_id.get(entry['video_id'])
        if (previous is None or previous.get('media', {}).get('relative_path') != entry.get('media', {}).get('relative_path')
                or previous.get('dataset') != entry.get('dataset') or previous.get('model_label') != entry.get('model_label')
                or previous.get('split') != entry.get('split')):
            raise ValueError(f'reuse manifest/source mismatch for {entry["video_id"]}')
    return stored

def _load_reuse_runs(eval_dir, selected, run_map=None, destination_root=None, data_root=DATA_ROOT):
    recorded_path = Path(eval_dir) / 'runs.json'
    recorded = json.loads(recorded_path.read_text()) if recorded_path.is_file() else None
    runs = []
    required = ('config.json', 'metadata.json', 'features.jsonl', 'violence.jsonl')
    for entry in selected:
        video_id = entry['video_id']
        candidates = []
        if run_map is not None and video_id in run_map:
            candidates = [Path(run_map[video_id])]
        elif isinstance(recorded, list):
            candidates = [Path(row['source_run']) for row in recorded if row.get('video_id') == video_id and row.get('status') == 'success']
        if len(candidates) != 1:
            raise ValueError(f'reuse requires exactly one saved source run for {video_id}; provide runs.json or EXISTING_RUN_MAP')
        original_directory = candidates[0].expanduser().resolve()
        missing = [name for name in required if not (original_directory / name).is_file()]
        if missing:
            raise ValueError(f'reuse source run for {video_id} is missing: {", ".join(missing)}')
        metadata = json.loads((original_directory / 'metadata.json').read_text())
        _validate_reuse_source_run(original_directory, entry, data_root)
        run_directory = original_directory
        if destination_root is not None:
            run_directory = (Path(destination_root) / video_id).resolve()
            run_directory.mkdir(parents=True, exist_ok=False)
            for name in required + ('metrics.json',):
                source_path = original_directory / name
                if source_path.is_file():
                    shutil.copy2(source_path, run_directory / name)
        runs.append({
            'video_id': video_id, 'status': 'success', 'source_run': str(run_directory),
            'source_run_id': metadata.get('run_id', original_directory.name),
            'evidence_run_id': metadata.get('run_id', original_directory.name),
            'source_run_original': str(original_directory),
        })
    return runs

def materialize_evaluation(manifest, config_path=CONFIG_PATH, data_root=DATA_ROOT, evaluation_root=EVAL_ROOT,
                           selected_ids=None, reuse=False, existing_evaluation_dir=None, existing_run_map=None, manifest_reviewed=False):
    require_manifest_ready(manifest)
    selected = [entry for entry in manifest['entries'] if selected_ids is None or entry['video_id'] in selected_ids]
    if not selected:
        raise ValueError('selected_ids did not select any manifest entries')
    if reuse:
        if existing_evaluation_dir is None:
            raise ValueError('REUSE_EXISTING_RUN requires EXISTING_EVALUATION_DIR')
        source_eval_dir = Path(existing_evaluation_dir).expanduser().resolve()
        if not source_eval_dir.is_dir():
            raise FileNotFoundError(f'reuse evaluation directory is missing: {source_eval_dir}')
        _validate_reuse_manifest(source_eval_dir, selected, data_root)
        eval_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-reuse-' + uuid4().hex[:8]
        eval_dir = (Path(evaluation_root) / eval_id).expanduser().resolve()
        source_root = eval_dir / 'source_runs'
        source_root.mkdir(parents=True, exist_ok=False)
        (eval_dir / 'evidence').mkdir(parents=True, exist_ok=True)
        evidence_root = source_eval_dir / 'evidence'
        base = load_config(config_path)
        config = replace(base, output_directory=source_root, m5=replace(base.m5, evidence_root=evidence_root))
        runs = _load_reuse_runs(source_eval_dir, selected, existing_run_map, source_root, data_root)
        failures = []
        (eval_dir / 'reuse_source.json').write_text(json.dumps({'source_evaluation_dir': str(source_eval_dir), 'read_only': True}, indent=2) + '\n')
        print(f'Reusing saved source artifacts from {source_eval_dir} into new evaluation {eval_dir}; process_video is disabled')
    else:
        eval_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid4().hex[:8]
        eval_dir = (Path(evaluation_root) / eval_id).expanduser().resolve()
        source_root = eval_dir / 'source_runs'
        evidence_root = eval_dir / 'evidence'
        source_root.mkdir(parents=True, exist_ok=False)
        evidence_root.mkdir(parents=True, exist_ok=True)
        base = load_config(config_path)
        config = replace(base, output_directory=source_root, m5=replace(base.m5, evidence_root=evidence_root))
        runs, failures = [], []
    (eval_dir / 'config.json').write_text(json.dumps({'manifest_matching': manifest['matching'], 'pipeline': config.as_dict(), 'reuse': reuse}, indent=2, sort_keys=True, default=str) + '\n')
    (eval_dir / 'manifest.json').write_text(json.dumps(manifest, indent=2, sort_keys=True) + '\n')
    revision = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=REPO, capture_output=True, text=True).stdout.strip() or None
    (eval_dir / 'environment.json').write_text(json.dumps({'python': sys.version, 'platform': platform.platform(), 'git_revision': revision, 'data_root': str(Path(data_root).resolve()), 'reuse': reuse}, indent=2, sort_keys=True, default=str) + '\n')
    predictions_by_strategy = {strategy: {} for strategy in STRATEGIES}
    if not reuse:
        for index, entry in enumerate(selected, start=1):
            print(f'[{index}/{len(selected)}] processing {entry["video_id"]}; detector + rolling violence inference may take a while', flush=True)
            try:
                media = resolve_media(entry, data_root, require_exists=True)
                result = process_video(config, input_override=media)
                run = {'video_id': entry['video_id'], 'status': 'success', 'source_run': str(result.run_directory), 'source_run_id': result.run_id}
                runs.append(run)
                print(f'[{index}/{len(selected)}] complete {entry["video_id"]}: {result.run_directory}', flush=True)
            except Exception as exc:
                failure = {'video_id': entry['video_id'], 'scenario': entry['scenario'], 'status': 'failed', 'error_type': type(exc).__name__, 'error': str(exc)}
                runs.append(failure); failures.append(failure)
                print(f'[{index}/{len(selected)}] failed {entry["video_id"]}: {failure["error"]}', flush=True)
    for run in runs:
        if run['status'] != 'success':
            continue
        entry = next(entry for entry in selected if entry['video_id'] == run['video_id'])
        try:
            run_directory = Path(run['source_run'])
            replay_config = config
            if reuse:
                stored_values = json.loads((run_directory / 'config.json').read_text()).get('config', {})
                replay_config = replace(
                    base,
                    input_path=Path(stored_values['input_path']),
                    output_directory=Path(stored_values['output_directory']),
                    m5=replace(base.m5, evidence_root=Path(stored_values['m5']['evidence_root'])),
                )
            replay_run(run_directory, replay_config, STRATEGIES)
            for strategy in STRATEGIES:
                predictions_by_strategy[strategy][entry['video_id']] = active_alerts(
                    entry['video_id'], strategy, run_directory / 'replay', run_directory,
                    evidence_root, entry['source_id'], run.get('evidence_run_id'))
        except Exception as exc:
            failure = {'video_id': entry['video_id'], 'scenario': entry['scenario'], 'status': 'failed', 'error_type': type(exc).__name__, 'error': str(exc)}
            failures.append(failure)
            run['status'] = 'failed'
            print(f'replay failed for {entry["video_id"]}: {failure["error"]}', flush=True)
    (eval_dir / 'runs.json').write_text(json.dumps(runs, indent=2, sort_keys=True) + '\n')
    successful_entries = [entry for entry in selected if any(run['video_id'] == entry['video_id'] and run['status'] == 'success' for run in runs)]
    strategy_metrics = {strategy: aggregate_strategy(successful_entries, predictions_by_strategy[strategy], strategy, manifest['matching']) for strategy in STRATEGIES}
    all_predictions = [row for strategy in STRATEGIES for rows in predictions_by_strategy[strategy].values() for row in rows]
    all_incidents = [{'strategy': row['strategy'], 'video_id': row['video_id'], **row} for row in all_predictions]
    write_jsonl(eval_dir / 'predictions.jsonl', all_predictions)
    write_jsonl(eval_dir / 'incidents.jsonl', all_incidents)
    failure_slices = failure_slice_rows(strategy_metrics, failures)
    latency_rows = [{'video_id': run['video_id'], **{key: json.loads((Path(run['source_run']) / 'metrics.json').read_text()).get(key) for key in ('total_seconds', 'effective_fps', 'violence_seconds', 'fusion_seconds')}} for run in runs if run['status'] == 'success' and (Path(run['source_run']) / 'metrics.json').is_file()]
    metrics_payload = {'strategies': {name: {key: value for key, value in metric.items() if key != 'entry_results'} for name, metric in strategy_metrics.items()}, 'stage_health': stage_health_summary(runs), 'latency': latency_rows, 'failures': failures, 'failure_slices': failure_slices}
    (eval_dir / 'metrics.json').write_text(json.dumps(metrics_payload, indent=2, sort_keys=True, default=str) + '\n')
    m3a_rows = model_predictions_m3a(runs, config.violence.threshold)
    model_rows = {'M3A': m3a_rows}
    m3b_path = os.environ.get('CROWD_SAFETY_M3B_PREDICTIONS'); model_status = 'pending-m3b'
    if m3b_path and Path(m3b_path).is_file():
        try: payload = json.loads(Path(m3b_path).read_text())
        except (OSError, ValueError): payload = {}
        if not isinstance(payload, dict): payload = {}
        target_ids = {entry['video_id'] for entry in selected if entry.get('model_label') in {'normal', 'violent'}}
        prediction_ids = [row.get('video_id') for row in payload.get('predictions', []) if isinstance(row, dict)]
        compatible = (payload.get('schema_version') == '1.0' and payload.get('model') == 'M3B' and payload.get('split') == 'test' and isinstance(payload.get('threshold'), (int, float)) and isinstance(payload.get('checkpoint'), str) and bool(payload.get('predictions')) and set(prediction_ids) == target_ids and len(prediction_ids) == len(set(prediction_ids)) and all(isinstance(row, dict) and row.get('video_id') in target_ids and row.get('status') in {'available', 'degraded', 'unavailable'} and (row.get('score') is None or isinstance(row.get('score'), (int, float))) for row in payload['predictions']))
        if compatible:
            model_rows['M3B'] = [{**row, 'model': 'M3B', 'checkpoint': payload['checkpoint'], 'split': payload['split'], 'threshold': payload['threshold']} for row in payload['predictions']]
            model_status = 'available'
        else: model_status = 'pending-incompatible-m3b'
    write_jsonl(eval_dir / 'model_predictions.jsonl', [row for rows in model_rows.values() for row in rows])
    vlm_rows = [vlm_disabled_record(row['incident_id']) for row in all_predictions]
    write_jsonl(eval_dir / 'vlm_reviews.jsonl', vlm_rows)
    positive_events = [event for entry in selected for event in _violence_events(entry)]
    m3a_available = [row for row in m3a_rows if row.get('status') == 'available']
    integrity = {
        'manifest_reviewed': bool(manifest_reviewed),
        'labelled_positive_event_count': len(positive_events),
        'true_duration_s': round(sum(float(event['end_s']) - float(event['onset_s']) for event in positive_events), 3),
        'camera_hours': round(sum(float(entry['duration_s']) for entry in selected) / 3600.0, 4),
        'source_run_failures': len(failures),
        'git_sha': revision,
        'm3a_checkpoint': m3a_available[0].get('checkpoint') if m3a_available else None,
        'm3a_revision': m3a_available[0].get('revision') if m3a_available else None,
        'm3a_threshold': config.violence.threshold,
        'm3b_status': model_status,
    }
    metrics_payload['evaluation_integrity'] = integrity
    model_metrics = render_report(eval_dir, manifest, strategy_metrics, model_rows, model_status, vlm_rows, failures, failure_slices, latency_rows, integrity)
    metrics_payload['models'] = {row['model']: row for row in model_metrics}
    (eval_dir / 'metrics.json').write_text(json.dumps(metrics_payload, indent=2, sort_keys=True, default=str) + '\n')
    return eval_dir, {'runs': runs, 'strategies': strategy_metrics, 'model_metrics': model_metrics, 'vlm_reviews': vlm_rows, 'failures': failures}


In [ ]:
# Prove the five-strategy replay layout without requiring media or a model.
from crowd_safety.config import load_config
with tempfile.TemporaryDirectory() as tmp:
    root = Path(tmp); source = root / 'source.mp4'; config_path = root / 'pipeline.toml'; artifact_dir = root / 'artifacts'
    config_path.write_text(f'[input]\npath = "{source}"\n[output]\ndirectory = "{artifact_dir}"\n[processing]\nresize = [16, 12]\n')
    config = load_config(config_path); run = root / 'run'; run.mkdir()
    values = resolved_config(config, config.input_path); (run / 'config.json').write_text(json.dumps({'config_hash': config_hash(values), 'config': values}))
    (run / 'features.jsonl').write_text(json.dumps({'features': [{'source_id': 'camera', 'roi_name': 'zone', 'timestamp_s': 1.0, 'status': 'available', 'occupancy': 2}]}) + '\n')
    (run / 'violence.jsonl').write_text(json.dumps({'evidence': {'source_id': 'camera', 'region_id': None, 'clip_start_s': 0.0, 'clip_end_s': 1.0, 'score': 0.9, 'model': 'fixture', 'revision': 'fixture', 'label_mapping': [['safe', 0], ['unsafe', 1]], 'status': 'available'}}) + '\n')
    replay_run(run, config, STRATEGIES)
    assert {path.name for path in (run / 'replay').iterdir()} == set(STRATEGIES) | {'metadata.json'}
print('synthetic replay self-check passed')

# Reuse safety checks: missing mappings fail before any model runner can be called.
with tempfile.TemporaryDirectory() as tmp:
    root = Path(tmp)
    (root / 'manifest.json').write_text(json.dumps({'entries': []}))
    try:
        _load_reuse_runs(root, [{'video_id': 'missing'}])
    except ValueError as exc:
        assert 'reuse requires exactly one saved source run' in str(exc)
    else:
        raise AssertionError('reuse accepted an unmapped source run')
    try:
        require_manifest_ready({'entries': [{'video_id': 'violent', 'model_label': 'violent', 'expected_alert': False, 'events': []}]})
    except ValueError as exc:
        assert 'Manifest review required' in str(exc)
    else:
        raise AssertionError('execution gate accepted an unfinished manifest')
print('reuse and readiness self-checks passed')


# A compatible fixture proves reuse reaches replay/report code without process_video.
with tempfile.TemporaryDirectory() as tmp:
    root = Path(tmp); eval_dir = root / 'evaluation'; source = root / 'source-run'; source.mkdir(parents=True)
    manifest = {'schema_version': '1.0', 'evaluation_id': 'reuse-fixture',
                'matching': {'temporal_tolerance_s': 0.0, 'require_temporal_overlap': True, 'actionable_state': 'active'},
                'entries': [{'video_id': 'normal', 'media': {'relative_path': 'clip.mp4'}, 'split': 'test', 'dataset': 'fixture',
                             'source_id': 'camera', 'session_id': 'session', 'scenario': 'normal', 'duration_s': 10.0,
                             'expected_alert': False, 'model_label': 'normal', 'events': [], 'tags': []}]}
    eval_dir.mkdir(); (eval_dir / 'manifest.json').write_text(json.dumps(manifest))
    (eval_dir / 'runs.json').write_text(json.dumps([{'video_id': 'normal', 'status': 'success', 'source_run': str(source)}]))
    base = load_config(CONFIG_PATH)
    source_eval_dir = eval_dir.resolve()
    eval_dir = source_eval_dir
    expected_config = replace(base, output_directory=eval_dir / 'source_runs', m5=replace(base.m5, evidence_root=eval_dir / 'evidence'))
    values = resolved_config(expected_config, root / 'clip.mp4')
    (source / 'config.json').write_text(json.dumps({'config_hash': config_hash(values), 'config': values}))
    (source / 'metadata.json').write_text(json.dumps({'run_id': 'fixture-run', 'provenance': {'violence_provenance': M3A_X3D_PROVENANCE}}))
    (source / 'metrics.json').write_text(json.dumps({'stage_health': {}}))
    (source / 'features.jsonl').write_text(json.dumps({'features': [{'source_id': 'camera', 'roi_name': 'zone', 'timestamp_s': 1.0, 'status': 'available', 'occupancy': 0}]}) + '\n')
    (source / 'violence.jsonl').write_text(json.dumps({'evidence': {'source_id': 'camera', 'region_id': None, 'clip_start_s': 0.0, 'clip_end_s': 1.0, 'score': None, 'model': 'fixture', 'revision': 'fixture', 'label_mapping': [], 'status': 'unavailable'}}) + '\n')
    previous_runner = process_video
    def _unexpected_runner(*args, **kwargs):
        raise AssertionError('reuse invoked process_video')
    process_video = _unexpected_runner
    reused_dir, reused = materialize_evaluation(manifest, data_root=root, reuse=True, existing_evaluation_dir=source_eval_dir, evaluation_root=root / 'outputs')
    process_video = previous_runner
    assert reused_dir != source_eval_dir and reused['runs'][0]['source_run_id'] == 'fixture-run'
    assert not (source / 'replay').exists() and (reused_dir / 'reuse_source.json').is_file()
print('compatible saved-artifact reuse self-check passed')


In [ ]:
model_fixture_entries = [
    {'video_id': 'normal', 'model_label': 'normal', 'events': []},
    {'video_id': 'violent', 'model_label': 'violent', 'events': [{'event_id': 'v1', 'label': 'violence', 'onset_s': 2.0, 'end_s': 4.0, 'expected_alert': True}]},
]
model_fixture_rows = [
    {'video_id': 'normal', 'clip_start_s': 0.0, 'clip_end_s': 1.0, 'score': 0.1, 'threshold': 0.5, 'status': 'available'},
    {'video_id': 'violent', 'clip_start_s': 1.0, 'clip_end_s': 1.5, 'score': 0.4, 'threshold': 0.5, 'status': 'available'},
    {'video_id': 'violent', 'clip_start_s': 1.0, 'clip_end_s': 3.0, 'timestamp_s': 3.0, 'score': 0.9, 'threshold': 0.5, 'status': 'available'},
]
model_fixture_metrics = score_model_predictions(model_fixture_rows, model_fixture_entries, {'temporal_tolerance_s': 0.0})
assert model_fixture_metrics['tp'] == 1 and model_fixture_metrics['tn'] == 2 and model_fixture_metrics['fp'] == 0 and model_fixture_metrics['fn'] == 0
assert model_fixture_metrics['event_metrics']['recall'] == 1.0 and model_fixture_metrics['event_metrics']['first_detection_delay_s_mean'] == 1.0
assert model_fixture_metrics['window_metrics']['false_positive_window_rate'] == 0.0
empty_metrics = score_model_predictions([], model_fixture_entries)
assert empty_metrics['status'] == 'pending-compatible-scores' and empty_metrics['event_metrics']['status'] == 'unavailable-no-compatible-windows'
zero_metrics = score_model_predictions([], [{'video_id': 'normal', 'model_label': 'normal', 'events': []}])
assert zero_metrics['event_metrics']['status'] == 'invalid-no-positive-events'
disabled_review = vlm_disabled_record('fixture-incident')
assert disabled_review['status'] == 'disabled' and disabled_review['grounded'] is None and disabled_review['contradicts_reasons'] is None
print('M3A window/event metrics and disabled-VLM self-checks passed')


with tempfile.TemporaryDirectory() as tmp:
    report_manifest = {'evaluation_id': 'report-fixture', 'matching': {'temporal_tolerance_s': 0.0}, 'entries': [
        {'video_id': 'normal', 'duration_s': 10.0, 'model_label': 'normal', 'events': []},
        {'video_id': 'violent', 'duration_s': 10.0, 'model_label': 'violent', 'events': [{'event_id': 'v1', 'label': 'violence', 'onset_s': 2.0, 'end_s': 4.0, 'expected_alert': True}]},
    ]}
    strategy = {'strategy': 'temporal', 'expected_events': 1, 'alerts': 1, 'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'false_alerts_per_camera_hour': 0.0, 'duplicate_alerts_per_true_event': 0.0, 'detection_delay_s_mean': 0.0, 'duration_error_s_mean': 0.0, 'evidence_completeness': None, 'evidence_completeness_status': 'not_comparable', 'entry_results': []}
    render_report(Path(tmp), report_manifest, {'temporal': strategy}, {'M3A': model_fixture_rows}, 'pending-m3b', [], [], [], [], {'manifest_reviewed': True, 'labelled_positive_event_count': 1, 'true_duration_s': 2.0, 'source_run_failures': 0, 'git_sha': 'fixture', 'm3a_threshold': 0.5})
    summary = (Path(tmp) / 'summary.md').read_text()
    assert 'M3A violence-model evaluation' in summary and 'Evaluation integrity' in summary and 'not_comparable' in summary and '{{' not in summary
print('report and evaluation-integrity self-check passed')


## 4. Execute the evaluation

Execution order for a real reviewed run:

1. Run bootstrap/configuration and set RUN_MANIFEST_REVIEW = True in the review section. Watch each model_label == "violent" clip, enter relative events, and require Manifest ready for evaluation: YES.
2. Set RUN_EVALUATION = True in the small execution-mode cell. For a smoke run, set SELECTED_VIDEO_IDS to one reviewed video.
3. Choose one path: leave REUSE_EXISTING_RUN = False for detector/violence inference, or set it to True with EXISTING_EVALUATION_DIR (and runs.json or EXISTING_RUN_MAP) to reuse saved features.jsonl, violence.jsonl, metadata, source evidence, and replay inputs. Reuse still runs deterministic replay, matching, metrics, and report generation; it never calls process_video.
4. Run the final execution cell. It repeats the readiness gate immediately before materialization, writes runs.json, and prints failures rather than hiding them.
5. Inspect metrics.json, summary.md, predictions.jsonl, model_predictions.jsonl, replay transitions, source evidence manifests, and the generated demo-review.json before expanding to the full manifest.

Leave RUN_EVALUATION = False for self-check mode. The notebook self-checks require no Drive media, Google credentials, or real model inference. M3B remains pending unless a compatible held-out export is explicitly supplied; VLM records are supplementary.


In [ ]:
from collections import defaultdict

BINARY_MANIFEST_FILES = {'train': 'train.json', 'validation': 'val.json', 'test': 'test.json', 'external_test': 'external_test.json'}

def _binary_manifest_path(relative, data_root=DATA_ROOT, require_exists=False):
    path = PurePosixPath(relative)
    if path.is_absolute() or '..' in path.parts or not relative.startswith('datasets/'):
        raise ValueError(f'unsafe Drive path: {relative}')
    root = Path(data_root).expanduser().resolve()
    candidate = (root / relative).resolve()
    try:
        candidate.relative_to(root)
    except ValueError as exc:
        raise ValueError(f'Drive path escapes data root: {relative}') from exc
    if require_exists and not candidate.is_file():
        raise FileNotFoundError(f'manifest media is missing: {candidate}')
    return candidate

def load_binary_manifests(manifest_root=DATA_ROOT / 'manifests', require_files=False):
    records, paths, hashes = [], set(), set()
    for split, filename in BINARY_MANIFEST_FILES.items():
        path = Path(manifest_root) / filename
        if not path.is_file():
            raise FileNotFoundError(f'missing binary manifest: {path}')
        payload = json.loads(path.read_text())
        if payload.get('manifest_type') != 'binary_video' or payload.get('split') != split or not isinstance(payload.get('records'), list):
            raise ValueError(f'invalid binary manifest contract: {path}')
        for record in payload['records']:
            if not isinstance(record, dict) or set(('dataset', 'relative_path', 'label', 'split')) - record.keys() or record['split'] != split or record['label'] not in {'normal', 'violent'}:
                raise ValueError(f'unsupported binary record: {record}')
            if record['dataset'] not in {'violent_flows', 'ubi_fights', 'surveillance_fight', 'scvd'}:
                dataset_name = record.get('dataset')
                raise ValueError(f'unsupported binary dataset: {dataset_name}')
            _binary_manifest_path(record['relative_path'], DATA_ROOT, require_files)
            digest = record.get('sha256')
            if record['relative_path'] in paths or (digest and digest in hashes):
                raise ValueError('binary manifest contains duplicate path or content hash')
            paths.add(record['relative_path']); hashes.add(digest) if digest else None
            if split == 'external_test' and record['dataset'] != 'violent_flows':
                raise ValueError('external_test may contain only violent_flows')
            if split != 'external_test' and record['dataset'] == 'violent_flows':
                raise ValueError('violent_flows may not enter train/validation/test')
            records.append(record)
    return records

def _binary_metrics(rows, threshold):
    scored = [row for row in rows if row.get('status') == 'available' and row.get('score') is not None]
    tp = sum(row['label'] == 'violent' and float(row['score']) >= threshold for row in scored)
    tn = sum(row['label'] == 'normal' and float(row['score']) < threshold for row in scored)
    fp = sum(row['label'] == 'normal' and float(row['score']) >= threshold for row in scored)
    fn = sum(row['label'] == 'violent' and float(row['score']) < threshold for row in scored)
    total = len(scored)
    return {'status': 'available' if scored else 'pending-compatible-scores', 'samples': total, 'unavailable_windows': len(rows) - total, 'tp': tp, 'tn': tn, 'fp': fp, 'fn': fn, 'precision': tp / (tp + fp) if tp + fp else 0.0, 'recall': tp / (tp + fn) if tp + fn else 0.0, 'f1': 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else 0.0, 'accuracy': (tp + tn) / total if total else None, 'confusion_matrix': [[tn, fp], [fn, tp]] if scored else None, 'score_distribution': {'min': min((float(row['score']) for row in scored), default=None), 'max': max((float(row['score']) for row in scored), default=None), 'mean': sum(float(row['score']) for row in scored) / total if total else None}, 'false_positives': [{'video_id': row['video_id'], 'score': row['score']} for row in scored if row['label'] == 'normal' and float(row['score']) >= threshold], 'false_negatives': [{'video_id': row['video_id'], 'score': row['score']} for row in scored if row['label'] == 'violent' and float(row['score']) < threshold], 'latency_ms_mean': sum(float(row['latency_ms']) for row in scored if row.get('latency_ms') is not None) / sum(row.get('latency_ms') is not None for row in scored) if any(row.get('latency_ms') is not None for row in scored) else None}

def score_component_rows(rows, threshold):
    grouped = defaultdict(list)
    for row in rows:
        grouped[(row['dataset'], row['split'])].append(row)
    return {f'{dataset}:{split}': {'dataset': dataset, 'split': split, 'threshold': threshold, **_binary_metrics(items, threshold)} for (dataset, split), items in sorted(grouped.items())}

def ubi_incident_entries(entries):
    return [entry for entry in entries if entry.get('dataset') == 'ubi_fights' and entry.get('split') == 'test']

def validate_m3a_reusable_run(run_directory, record):
    run_directory = Path(run_directory)
    metadata = json.loads((run_directory / 'metadata.json').read_text())
    provenance = metadata.get('provenance', {}).get('violence_provenance')
    if not isinstance(provenance, dict):
        raise ValueError('saved M3A run has no violence provenance')
    if any(provenance.get(key) != value for key, value in M3A_X3D_PROVENANCE.items()):
        raise ValueError('saved M3A run provenance does not match the pinned X3D contract')
    config = json.loads((run_directory / 'config.json').read_text()).get('config', {})
    stored_input = config.get('input_path')
    expected_input = _binary_manifest_path(record['relative_path'])
    if not isinstance(stored_input, str) or Path(stored_input).expanduser().resolve() != expected_input:
        raise ValueError('saved M3A run input does not match the manifest media')

def run_m3a_component_stage(records, config_path=CONFIG_PATH, run_map=None):
    selected = [record for record in records if record['split'] in {'test', 'external_test'}]
    base = load_config(config_path)
    threshold = base.violence.threshold
    run_map = run_map or {}
    rows = []
    for index, record in enumerate(selected, start=1):
        run_directory = Path(run_map[record['relative_path']]) if record['relative_path'] in run_map else None
        if run_directory is not None:
            validate_m3a_reusable_run(run_directory, record)
        if run_directory is None:
            media = _binary_manifest_path(record['relative_path'], DATA_ROOT, require_exists=True)
            run_config = replace(base, input_path=media, output_directory=EVAL_ROOT / 'm3a_component_runs')
            relative_path = record['relative_path']
            print(f'[{index}/{len(selected)}] processing {relative_path}', flush=True)
            run_directory = process_video(run_config, input_override=media).run_directory
        raw = model_predictions_m3a([{'video_id': record['relative_path'], 'status': 'success', 'source_run': str(run_directory)}], threshold)
        rows.extend({**row, 'video_id': record['relative_path'], 'dataset': record['dataset'], 'label': record['label'], 'split': record['split']} for row in raw)
    component_root = EVAL_ROOT / 'm3a_component'
    component_root.mkdir(parents=True, exist_ok=True)
    metrics = {'threshold': threshold, 'threshold_source': 'development_config_only; external_test was not calibrated', 'by_dataset': score_component_rows(rows, threshold)}
    write_jsonl(component_root / 'predictions.jsonl', rows)
    (component_root / 'metrics.json').write_text(json.dumps(metrics, indent=2, sort_keys=True) + '\n')
    return metrics

def select_ubi_incident_manifest(path=MANIFEST_PATH, require_files=False):
    incident_manifest = load_manifest(path, DATA_ROOT, require_files=require_files)
    selected = ubi_incident_entries(incident_manifest['entries'])
    if not selected:
        print('Stage 2 skipped: no reviewed UBI-Fights test entries in the incident manifest')
        return None
    result = deepcopy(incident_manifest)
    result['evaluation_id'] = str(result.get('evaluation_id', 'evaluation')) + '-ubi-test'
    result['entries'] = selected
    require_manifest_ready(result)
    return result

def run_drive_evaluation():
    records = load_binary_manifests(require_files=True)
    run_map = json.loads(os.environ['CROWD_SAFETY_M3A_RUN_MAP']) if os.environ.get('CROWD_SAFETY_M3A_RUN_MAP') else None
    stage1 = run_m3a_component_stage(records, run_map=run_map)
    incident_manifest = select_ubi_incident_manifest(require_files=True)
    stage2 = None
    if incident_manifest is not None:
        stage2_dir, stage2 = materialize_evaluation(incident_manifest, selected_ids=SELECTED_VIDEO_IDS, reuse=REUSE_EXISTING_RUN, existing_evaluation_dir=EXISTING_EVALUATION_DIR, existing_run_map=EXISTING_RUN_MAP, manifest_reviewed=has_review_provenance(MANIFEST_PATH, incident_manifest))
        print('Stage 2 evaluation directory:', stage2_dir)
    print('Stage 1 per-dataset metrics:', json.dumps(stage1, indent=2, sort_keys=True))
    return {'stage1': stage1, 'stage2': stage2}

_m6_fixture = [{'dataset': 'scvd', 'split': 'test', 'label': 'normal', 'score': 0.1, 'status': 'available'}, {'dataset': 'scvd', 'split': 'test', 'label': 'violent', 'score': 0.9, 'status': 'available'}, {'dataset': 'violent_flows', 'split': 'external_test', 'label': 'violent', 'score': None, 'status': 'unavailable'}]
_m6_metrics = score_component_rows(_m6_fixture, 0.4)
assert _m6_metrics['scvd:test']['accuracy'] == 1.0 and _m6_metrics['violent_flows:external_test']['unavailable_windows'] == 1
assert [entry['video_id'] for entry in ubi_incident_entries([{'video_id': 'ubi', 'dataset': 'ubi_fights', 'split': 'test'}, {'video_id': 'other', 'dataset': 'scvd', 'split': 'test'}])] == ['ubi']
print('M6 binary scoring/external isolation/UBI selection self-check: ok')

def start_ephemeral_review(source_run, config):
    from fastapi.testclient import TestClient
    from crowd_safety.api import create_app
    store = MemoryPersistence(); imported = import_run(source_run, store, config.m5.evidence_root)
    client = TestClient(create_app(store, config.m5.evidence_root))
    health = client.get('/health').json()
    records = store.list_incidents()
    if not records: return {'imported': imported, 'health': health, 'status': 'no_incident_to_review'}
    incident_id = records[0]['incident']['incident_id']
    response = client.post(f'/incidents/{incident_id}/acknowledge', json={'actor': 'colab-reviewer', 'timestamp': datetime.now(timezone.utc).isoformat(), 'note': 'M6 offline demo review'})
    response.raise_for_status()
    return {'imported': imported, 'health': health, 'status': 'acknowledged', 'action': response.json()}

if False and RUN_EVALUATION:
    manifest = load_manifest(MANIFEST_PATH, DATA_ROOT, require_files=True)
    require_manifest_ready(manifest)
    if REUSE_EXISTING_RUN and EXISTING_EVALUATION_DIR is None:
        raise ValueError('Set EXISTING_EVALUATION_DIR before enabling REUSE_EXISTING_RUN')
    evaluation_dir, result = materialize_evaluation(
        manifest,
        selected_ids=SELECTED_VIDEO_IDS,
        reuse=REUSE_EXISTING_RUN,
        existing_evaluation_dir=EXISTING_EVALUATION_DIR,
        existing_run_map=EXISTING_RUN_MAP,
        manifest_reviewed=has_review_provenance(MANIFEST_PATH, manifest),
    )
    successful = next((run for run in result['runs'] if run['status'] == 'success'), None)
    if successful:
        base_config = load_config(CONFIG_PATH); review_config = replace(base_config, m5=replace(base_config.m5, evidence_root=Path(evaluation_dir) / 'evidence'))
        review = start_ephemeral_review(successful['source_run'], review_config)
        (Path(evaluation_dir) / 'demo-review.json').write_text(json.dumps(review, indent=2, sort_keys=True, default=str) + '\n')
    print('evaluation directory:', evaluation_dir)
    print('completed:', len([run for run in result['runs'] if run['status'] == 'success']), '/', len(result['runs']))
    print('failures:', result['failures'])
if not RUN_EVALUATION:
    print('self-check mode; set RUN_EVALUATION = True for authorised Drive execution')

if RUN_EVALUATION:
    evaluation_result = run_drive_evaluation()
